In [10]:
import os
import re
import pandas as pd

In [29]:
input_path = "/content/drive/MyDrive/earbud_review_intelligence/data/raw/earbuds_reviews.csv"
output_path = "/content/drive/MyDrive/earbud_review_intelligence/data/processed/earbuds_reviews_clean.csv"
min_lex_review = 15

In [14]:
def is_probably_english(text:str) -> bool:
  #checks the text is ascii/punctuation with validating with 0.9
  if not text:
    return False
  ascii_char = sum(1 for c in text if ord(c) < 128)
  return(ascii_char / max(len(text),1)) > 0.9


In [23]:
def clean_dataframe(df:pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  #this applys all data cleaning procedure to make the data good to use
  #dropped duplicates
  before = len(df)
  df = df.drop_duplicates(subset=["asin","review_text"])
  print(f"dropped {before - len(df)} duplicates review")

  #drops the empty or very short reviews
  df["review_text"] = df["review_text"].fillna("").astype(str)
  df = df[df["review_text"].str.len() >= min_lex_review]

  #keeps only likely is english
  df = df[df["review_text"].apply(is_probably_english)]

  #Normalize whitespace once more
  df["review_text"] = df["review_text"].apply(lambda t:  re.sub(r"\s+"," ",t).strip())

  #convert timestamp to real Datetime
  df["review_date"] = pd.to_datetime(df["timestamp"],unit="ms", errors="coerce")

  #drops where rating is missing
  df = df.dropna(subset=["rating"])
  df["rating"] = df["rating"].astype(float)

  df = df.reset_index(drop=True)
  return df

In [18]:
def main():
  print(f"Loading the raw data")
  df = pd.read_csv(input_path)

  df_clean = clean_dataframe(df)

  print(f"\n{len(df_clean)} reviews remain after cleaning "
          f"({len(df) - len(df_clean)} removed)")
  df_clean.to_csv(output_path,index=False)
  print(f"saved cleaned data to {output_path}")

In [30]:
if __name__ == "__main__":
  main()


Loading the raw data
dropped 60 duplicates review

19867 reviews remain after cleaning (133 removed)
saved cleaned data to /content/drive/MyDrive/earbud_review_intelligence/data/processed/earbuds_reviews_clean.csv
